# MediScan AI — training on Google Colab (T4 GPU)

Produces **`trained_outputs.zip`** in the exact format `scripts/import_artifacts.py` expects
(`models/<key>/{model.onnx, best_model.pth, metadata.json}` + SHA-256 manifest).

Runtime → Change runtime type → **T4 GPU**. Metrics printed here are measured on this repository's
small held-out test split; they are **not** clinical validation.

In [ ]:
# 1. Get the code + data (upload the repo zip, or clone your fork)
# !git clone https://github.com/<you>/<repo>.git mediscan && %cd mediscan
import os; print(os.getcwd())

In [ ]:
# 2. Dependencies (Colab already has torch/torchvision)
!pip install -q onnx onnxruntime scikit-learn matplotlib pandas pillow

In [ ]:
# 3. Train, evaluate (threshold chosen on the VALIDATION split), export ONNX — per disease
for d in ["tb", "dr", "oral"]:
    !python -m src.train --disease {d}
    !python -m src.evaluate --disease {d}
    !python -m src.export_onnx --disease {d}

### Optional: gatekeeper (modality classifier)
Needs a folder of **unsupported** images for the `other` class (natural photos, documents, CT/MRI, skin photos…).
Without a gatekeeper the app still works, but automatic routing is disabled (manual module selection only).

In [ ]:
# 4. Gatekeeper (optional)
# !python -m src.prepare_gatekeeper --other-dir /content/other_images
# !python -m src.train --disease gatekeeper
# !python -m src.evaluate --disease gatekeeper
# !python -m src.export_onnx --disease gatekeeper

In [ ]:
# 5. Package → trained_outputs.zip (metadata.json + SHA-256 manifest written automatically)
# The bundled DR images are Gaussian-filtered; keep the default --dr-input-domain unless you trained on raw photos.
!python scripts/package_artifacts.py --out trained_outputs.zip
!python scripts/import_artifacts.py --archive trained_outputs.zip --dry-run   # validate before downloading

In [ ]:
# 6. Download
from google.colab import files
files.download("trained_outputs.zip")